In [0]:
output_customer_rfm_final_path = 's3://ims-ecom-stage/customer_rfm/'

In [0]:
customer_rfm_df = spark.sql("""
    SELECT
        o.customer_id,
        DATEDIFF(DATE('2018-10-17'), DATE(MAX(o.order_purchase_timestamp))) AS recency,
        COUNT(distinct o.order_id) AS frequency,
        SUM(oi.price) AS monetary
    FROM ecommerce.staging.orders o
    JOIN ecommerce.staging.order_items oi
        ON o.order_id = oi.order_id
    GROUP BY o.customer_id
""")

customer_rfm_df.createOrReplaceTempView("customer_rfm")

In [0]:
customer_rfm_final_df = spark.sql("""
SELECT
    customer_id,
    recency,
    frequency,
    monetary,
    6 - NTILE(5) OVER (ORDER BY recency ASC) AS rm_recency_score,
    NTILE(5) OVER (ORDER BY monetary ASC) AS rm_monetary_score,
    CASE
        WHEN (6 - NTILE(5) OVER (ORDER BY recency ASC)) >= 4
         AND NTILE(5) OVER (ORDER BY monetary ASC) >= 4
            THEN 'High-Value Recent'

        WHEN (6 - NTILE(5) OVER (ORDER BY recency ASC)) >= 4
         AND NTILE(5) OVER (ORDER BY monetary ASC) <= 3
            THEN 'Recent Low-Spender'

        WHEN (6 - NTILE(5) OVER (ORDER BY recency ASC)) <= 3
         AND NTILE(5) OVER (ORDER BY monetary ASC) >= 4
            THEN 'At-Risk High-Value'

        ELSE 'Low-Value Inactive'
    END AS customer_segment
FROM customer_rfm
""")


In [0]:
customer_rfm_final_df.write \
    .format('delta')  \
    .mode("overwrite") \
    .save(output_customer_rfm_final_path)

In [0]:
  %sql
CREATE TABLE IF NOT EXISTS ecommerce.staging.customer_rfm
USING delta
LOCATION 's3://ims-ecom-stage/customer_rfm/';

In [0]:
%sql
select * from ecommerce.staging.customer_rfm

customer_id,recency,frequency,monetary,rm_recency_score,rm_monetary_score,customer_segment
4b7decb9b58e2569548b8b4c8e20e8d7,44,1,145.0,5,4,High-Value Recent
10a79ef2783cae3d8d678e85fde235ac,49,1,6.9,5,1,Recent Low-Spender
496630b6740bcca28fce9ba50d8a26ef,49,1,24.9,5,1,Recent Low-Spender
6e353700bc7bcdf6ebc15d6de16d7002,49,1,45.9,5,2,Recent Low-Spender
e450a297a7bc6839ceb0cf1a2377fa02,49,1,63.9,5,2,Recent Low-Spender
c4c66f47534e09a03fc7a878a9eda5ea,49,1,65.0,5,2,Recent Low-Spender
898b7fee99c4e42170ab69ba59be0a8b,49,1,84.99,5,3,Recent Low-Spender
54365416b7ef5599f54a6c7821d5d290,49,1,99.0,5,3,Recent Low-Spender
56b1ac2855cc6d7950b4ffa6a9b41b0d,49,1,109.9,5,4,High-Value Recent
b8c19e70d00f6927388e4f31c923d785,49,1,169.8,5,4,High-Value Recent
